#Results YOLOv8

##Components Only (PCB-MC-C)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV8/components_only")
#RESULTS_BASE =Path("/content/drive/MyDrive/PCB_MC/resultsyolov8/components_only")
FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv8/YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [ ]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.3887,0.2688,0.6780,0.3621,0.4721,0.6379,results.csv
1,fold_1,0.3693,0.2478,0.4869,0.4213,0.4517,0.5787,results.csv
2,fold_2,0.2788,0.1853,0.4796,0.3156,0.3807,0.6844,results.csv
3,fold_3,0.3496,0.2439,0.6970,0.3434,0.4602,0.6565,results.csv
4,fold_4,0.4846,0.3372,0.5512,0.5102,0.5299,0.4898,results.csv
5,**AVERAGE**,0.3742,0.2566,0.5785,0.3905,0.4589,0.6095,-
6,**STD DEV**,0.0744,0.0547,0.1035,0.0773,0.0533,0.0773,-


##Full Dataset (PCB-MC-A)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV8/full_dataset")
FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv8/YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [ ]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.3193,0.2229,0.4244,0.3473,0.3820,0.6527,results.csv
1,fold_1,0.2916,0.1958,0.4181,0.3166,0.3603,0.6834,results.csv
2,fold_2,0.2064,0.1394,0.4819,0.2421,0.3223,0.7579,results.csv
3,fold_3,0.2753,0.1929,0.5378,0.2919,0.3784,0.7081,results.csv
4,fold_4,0.3490,0.2403,0.5203,0.3460,0.4156,0.6540,results.csv
5,**AVERAGE**,0.2883,0.1983,0.4765,0.3088,0.3717,0.6912,-
6,**STD DEV**,0.0537,0.0383,0.0544,0.0438,0.0341,0.0438,-


##Missing Only (PCB-MC-M)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV8/missing_only")
FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv8/YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [ ]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.1278,0.0607,0.2474,0.1309,0.1712,0.8691,results.csv
1,fold_1,0.0613,0.0290,0.1927,0.0762,0.1093,0.9237,results.csv
2,fold_2,0.0816,0.0434,0.4309,0.0848,0.1418,0.9152,results.csv
3,fold_3,0.0941,0.0499,0.4414,0.1091,0.1750,0.8909,results.csv
4,fold_4,0.0678,0.0345,0.5281,0.0723,0.1272,0.9277,results.csv
5,**AVERAGE**,0.0865,0.0435,0.3681,0.0947,0.1449,0.9053,-
6,**STD DEV**,0.0263,0.0125,0.1416,0.0248,0.0282,0.0248,-


##Non Missing (PCB-MC-P)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV8/non_missing")
FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv8/YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [ ]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"
    df = pd.concat([df, pd.DataFrame([avg])], ignore_index=True)

    display(df)

,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.31200,0.2024,0.44620,0.32320,0.37490,0.67680,results.csv
1,fold_1,0.54230,0.3908,0.59230,0.53700,0.56330,0.46300,results.csv
2,fold_2,0.47280,0.3213,0.71130,0.44020,0.54390,0.55970,results.csv
3,fold_3,0.45360,0.3356,0.62230,0.40280,0.48900,0.59720,results.csv
4,fold_4,0.20940,0.1264,0.26710,0.21900,0.24070,0.78110,results.csv
5,**AVERAGE**,0.39802,0.2753,0.52784,0.38444,0.44236,0.61556,-


#Results YOLOv11

##Components Only (PCB-MC-C)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV11/components_only")

FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [ ]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.4188,0.2897,0.6049,0.4367,0.5072,0.5633,results.csv
1,fold_1,0.4722,0.3090,0.6005,0.4647,0.5239,0.5353,results.csv
2,fold_2,0.2837,0.1961,0.3916,0.3456,0.3672,0.6544,results.csv
3,fold_3,0.3984,0.2693,0.7154,0.4025,0.5151,0.5975,results.csv
4,fold_4,0.5278,0.3795,0.7003,0.5125,0.5919,0.4875,results.csv
5,**AVERAGE**,0.4202,0.2887,0.6025,0.4324,0.5011,0.5676,-
6,**STD DEV**,0.0914,0.0663,0.1292,0.0631,0.0820,0.0631,-


##Full Dataset (PCB-MC-A)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV11/full_dataset")
FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv8/YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [ ]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.3379,0.2390,0.4318,0.3732,0.4004,0.6268,results.csv
1,fold_1,0.3065,0.1950,0.4151,0.3512,0.3805,0.6488,results.csv
2,fold_2,0.2309,0.1488,0.3341,0.2706,0.2990,0.7294,results.csv
3,fold_3,0.2752,0.1895,0.5548,0.2827,0.3745,0.7173,results.csv
4,fold_4,0.3716,0.2675,0.6471,0.3810,0.4796,0.6190,results.csv
5,**AVERAGE**,0.3044,0.2080,0.4766,0.3317,0.3868,0.6683,-
6,**STD DEV**,0.0545,0.0461,0.1238,0.0516,0.0646,0.0516,-


##Missing Only (PCB-MC-M)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV11/missing_only")
FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv8/YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [ ]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.1257,0.0582,0.1768,0.1793,0.1781,0.8207,results.csv
1,fold_1,0.0639,0.0281,0.4420,0.0911,0.1511,0.9089,results.csv
2,fold_2,0.0740,0.0321,0.3812,0.1382,0.2029,0.8618,results.csv
3,fold_3,0.0873,0.0395,0.2849,0.1205,0.1694,0.8795,results.csv
4,fold_4,0.0619,0.0275,0.0674,0.1246,0.0875,0.8753,results.csv
5,**AVERAGE**,0.0826,0.0371,0.2705,0.1307,0.1578,0.8692,-
6,**STD DEV**,0.0261,0.0127,0.1516,0.0321,0.0435,0.0321,-


##Non Missing (PCB-MC-P)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV11/non_missing")
FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv8/YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [ ]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.3310,0.2133,0.4326,0.3277,0.3729,0.6723,results.csv
1,fold_1,0.5755,0.4067,0.7399,0.5061,0.6011,0.4939,results.csv
2,fold_2,0.5466,0.3692,0.6368,0.5113,0.5672,0.4887,results.csv
3,fold_3,0.4372,0.3151,0.6143,0.4330,0.5080,0.5670,results.csv
4,fold_4,0.2593,0.1567,0.3220,0.3002,0.3107,0.6998,results.csv
5,**AVERAGE**,0.4299,0.2922,0.5491,0.4157,0.4720,0.5843,-
6,**STD DEV**,0.1358,0.1051,0.1684,0.0984,0.1254,0.0984,-


#Results YOLOv26

##Components Only (PCB-MC-C)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV26/components_only")

FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [ ]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.3981,0.2773,0.5958,0.3945,0.4747,0.6055,results.csv
1,fold_1,0.3797,0.2647,0.4901,0.3938,0.4367,0.6062,results.csv
2,fold_2,0.2941,0.1979,0.5923,0.3044,0.4021,0.6956,results.csv
3,fold_3,0.3364,0.2364,0.6669,0.3510,0.4599,0.6490,results.csv
4,fold_4,0.4778,0.3359,0.6214,0.5066,0.5582,0.4934,results.csv
5,**AVERAGE**,0.3772,0.2624,0.5933,0.3901,0.4663,0.6099,-
6,**STD DEV**,0.0692,0.0512,0.0649,0.0750,0.0582,0.0750,-


##Full Dataset (PCB-MC-A)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV26/full_dataset")
FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv8/YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [ ]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.2656,0.1832,0.4296,0.2895,0.3459,0.7105,results.csv
1,fold_1,0.2708,0.1807,0.3737,0.3273,0.3490,0.6727,results.csv
2,fold_2,0.2527,0.1743,0.3197,0.3039,0.3116,0.6961,results.csv
3,fold_3,0.3177,0.2264,0.5786,0.3203,0.4124,0.6797,results.csv
4,fold_4,0.3215,0.2207,0.4864,0.3329,0.3953,0.6671,results.csv
5,**AVERAGE**,0.2857,0.1971,0.4376,0.3148,0.3628,0.6852,-
6,**STD DEV**,0.0317,0.0245,0.1004,0.0178,0.0407,0.0178,-


##Missing Only (PCB-MC-M)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV26/missing_only")
FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv8/YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [ ]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.1710,0.0837,0.2601,0.2068,0.2304,0.7932,results.csv
1,fold_1,0.0636,0.0334,0.4052,0.0689,0.1178,0.9311,results.csv
2,fold_2,0.1187,0.0551,0.3965,0.1318,0.1978,0.8682,results.csv
3,fold_3,0.0632,0.0318,0.0811,0.1559,0.1067,0.8441,results.csv
4,fold_4,0.0802,0.0388,0.4090,0.0726,0.1233,0.9274,results.csv
5,**AVERAGE**,0.0993,0.0486,0.3104,0.1272,0.1552,0.8728,-
6,**STD DEV**,0.0460,0.0217,0.1425,0.0582,0.0553,0.0582,-


##Non Missing (PCB-MC-P)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV26/non_missing")
FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv8/YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [ ]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.3140,0.2143,0.4551,0.3033,0.3640,0.6967,results.csv
1,fold_1,0.4725,0.3395,0.6250,0.4292,0.5089,0.5708,results.csv
2,fold_2,0.5135,0.3527,0.6557,0.4593,0.5402,0.5407,results.csv
3,fold_3,0.3809,0.2798,0.6147,0.3947,0.4807,0.6053,results.csv
4,fold_4,0.2369,0.1537,0.3352,0.2461,0.2838,0.7539,results.csv
5,**AVERAGE**,0.3836,0.2680,0.5371,0.3665,0.4355,0.6335,-
6,**STD DEV**,0.1131,0.0842,0.1372,0.0892,0.1078,0.0892,-


#Results RT-DETR

##Components Only (PCB-MC-C)

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/RT-DETR/components_only")

FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [2]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.3811,0.2653,0.5848,0.4118,0.4833,0.5882,results.csv
1,fold_1,0.4123,0.2836,0.5021,0.4574,0.4787,0.5426,results.csv
2,fold_2,0.2784,0.1857,0.3547,0.3301,0.3420,0.6699,results.csv
3,fold_3,0.3291,0.2286,0.5790,0.3443,0.4318,0.6557,results.csv
4,fold_4,0.5177,0.3653,0.6549,0.5232,0.5817,0.4768,results.csv
5,**AVERAGE**,0.3837,0.2657,0.5351,0.4134,0.4635,0.5866,-
6,**STD DEV**,0.0906,0.0671,0.1144,0.0802,0.0871,0.0802,-


##Full Dataset (PCB-MC-A)

In [3]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/RT-DETR/full_dataset")
FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv8/YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [4]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.2502,0.1676,0.4562,0.2865,0.3519,0.7135,results.csv
1,fold_1,0.3143,0.2149,0.4977,0.3441,0.4069,0.6559,results.csv
2,fold_2,0.1995,0.1340,0.3740,0.2307,0.2854,0.7693,results.csv
3,fold_3,0.3043,0.2039,0.5616,0.3059,0.3960,0.6941,results.csv
4,fold_4,0.3576,0.2520,0.5163,0.3743,0.4340,0.6257,results.csv
5,**AVERAGE**,0.2852,0.1945,0.4812,0.3083,0.3748,0.6917,-
6,**STD DEV**,0.0613,0.0453,0.0709,0.0551,0.0581,0.0551,-


##Missing Only (PCB-MC-M)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/RT-DETR/missing_only")
FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv8/YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [ ]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.0847,0.0395,0.2425,0.1203,0.1608,0.8797,results.csv
1,fold_1,0.0190,0.0088,0.2240,0.0416,0.0702,0.9584,results.csv
2,fold_2,0.0463,0.0188,0.2553,0.1202,0.1635,0.8798,results.csv
3,fold_3,0.0431,0.0218,0.3063,0.0635,0.1052,0.9365,results.csv
4,fold_4,0.0120,0.0038,0.3316,0.0308,0.0564,0.9692,results.csv
5,**AVERAGE**,0.0410,0.0185,0.2719,0.0753,0.1112,0.9247,-
6,**STD DEV**,0.0286,0.0138,0.0452,0.0427,0.0498,0.0427,-


##Non Missing (PCB-MC-P)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


RESULTS_BASE = Path("/content/drive/MyDrive/PCB_MC/Results/RT-DETR/non_missing")
FOLDS = [f"fold_{i}" for i in range(5)]

def parse_ultralytics_results_csv(csv_path: Path, fold_name: str):
    """
    Parses Ultralytics results.csv and returns the last-epoch (or best available) metrics.
    Works for YOLOv8/YOLOv11 training logs.
    """
    if not csv_path.exists():
        return None

    df = pd.read_csv(csv_path)
    if df.empty:
        return None

    # Usually metrics are in the last row
    last = df.iloc[-1].to_dict()

    # Common Ultralytics column names (may vary slightly by version)
    # Train time columns omitted.
    map50 = last.get("metrics/mAP50(B)", np.nan)
    map5095 = last.get("metrics/mAP50-95(B)", np.nan)
    prec = last.get("metrics/precision(B)", np.nan)
    rec  = last.get("metrics/recall(B)", np.nan)

    # Some versions use different keys
    if np.isnan(map50):
        map50 = last.get("metrics/mAP50", np.nan)
    if np.isnan(map5095):
        map5095 = last.get("metrics/mAP50-95", np.nan)
    if np.isnan(prec):
        prec = last.get("metrics/precision", np.nan)
    if np.isnan(rec):
        rec  = last.get("metrics/recall", np.nan)

    if np.isnan(map50) and np.isnan(map5095) and np.isnan(prec) and np.isnan(rec):
        # If columns didn't match, show available columns for debugging
        raise ValueError(f"Unexpected columns in {csv_path}. Columns: {list(pd.read_csv(csv_path).columns)}")

    f1 = (2 * prec * rec) / (prec + rec + 1e-9) if (not np.isnan(prec) and not np.isnan(rec)) else np.nan
    fnr = 1 - rec if not np.isnan(rec) else np.nan

    return {
        "Fold": fold_name,
        "mAP@0.5": float(map50) if not np.isnan(map50) else np.nan,
        "mAP@0.5:0.95": float(map5095) if not np.isnan(map5095) else np.nan,
        "Precision": float(prec) if not np.isnan(prec) else np.nan,
        "Recall": float(rec) if not np.isnan(rec) else np.nan,
        "F1-Score": float(f1) if not np.isnan(f1) else np.nan,
        "FNR": float(fnr) if not np.isnan(fnr) else np.nan,
        "Source": "results.csv"
    }
def parse_ultralytics_results_txt(txt_path: Path, fold_name: str):
    """
    Fallback: tries to parse results.txt (older Ultralytics output).
    Looks for a line containing 'metrics' like 'mAP50' etc.
    """
    if not txt_path.exists():
        return None

    lines = txt_path.read_text().splitlines()
    if not lines:
        return None

    # Try to find the last line that contains mAP50
    target = None
    for line in reversed(lines):
        if "mAP50" in line and "precision" in line and "recall" in line:
            target = line
            break

    if target is None:
        return None

    # This is heuristic because formats vary.
    # You can print(target) if parsing fails and adjust.
    try:
        # Example expected tokens: precision recall mAP50 mAP50-95
        tokens = target.replace(",", " ").split()
        floats = [float(t) for t in tokens if t.replace(".", "", 1).isdigit()]

        # Heuristic mapping: last 4 floats correspond to P, R, mAP50, mAP50-95
        prec, rec, map50, map5095 = floats[-4], floats[-3], floats[-2], floats[-1]

        f1 = (2 * prec * rec) / (prec + rec + 1e-9)
        fnr = 1 - rec

        return {
            "Fold": fold_name,
            "mAP@0.5": map50,
            "mAP@0.5:0.95": map5095,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "FNR": fnr,
            "Source": "results.txt"
        }
    except:
        return None



In [ ]:
results = []

for fold in FOLDS:
    fold_dir = RESULTS_BASE / fold

    # Ultralytics usually saves results.csv inside the fold dir
    csv_path = fold_dir / "results.csv"
    txt_path = fold_dir / "results.txt"

    row = parse_ultralytics_results_csv(csv_path, fold)
    if row is None:
        row = parse_ultralytics_results_txt(txt_path, fold)

    if row is None:
        print(f"⚠️ Could not find metrics for {fold} in {fold_dir}")
    else:
        # Round for display
        for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
            if k in row and row[k] is not None and not np.isnan(row[k]):
                row[k] = round(row[k], 4)
        results.append(row)

df = pd.DataFrame(results)

if df.empty:
    print("❌ No results found. Check RESULTS_BASE and folder structure.")
else:
    avg = df.mean(numeric_only=True).to_dict()
    std = df.std(numeric_only=True).to_dict()

    avg["Fold"] = "**AVERAGE**"
    avg["Source"] = "-"

    std["Fold"] = "**STD DEV**"
    std["Source"] = "-"

    # Round average and std dev
    for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1-Score", "FNR"]:
        if k in avg and not pd.isna(avg[k]):
            avg[k] = round(avg[k], 4)
        if k in std and not pd.isna(std[k]):
            std[k] = round(std[k], 4)

    df = pd.concat([df, pd.DataFrame([avg, std])], ignore_index=True)

    display(df)


,Fold,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1-Score,FNR,Source
0,fold_0,0.2955,0.1889,0.5591,0.3075,0.3968,0.6925,results.csv
1,fold_1,0.4439,0.3015,0.6199,0.4702,0.5348,0.5298,results.csv
2,fold_2,0.4118,0.2727,0.6006,0.4321,0.5026,0.5679,results.csv
3,fold_3,0.3804,0.2667,0.5771,0.4036,0.4750,0.5964,results.csv
4,fold_4,0.1895,0.1078,0.4191,0.2162,0.2853,0.7838,results.csv
5,**AVERAGE**,0.3442,0.2275,0.5552,0.3659,0.4389,0.6341,-
6,**STD DEV**,0.1026,0.0789,0.0795,0.1031,0.0999,0.1031,-



#Visualization YOLOV11




In [ ]:
!pip install ultralytics --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.6 MB/s eta 0:00:00


In [ ]:
def read_yolo_labels(label_file_path, img_width, img_height):
    """
    Reads a YOLO format .txt label file and converts normalized bounding box
    coordinates to absolute pixel coordinates.
    Returns a list of ground truth boxes in the format [class_id, x1, y1, x2, y2].
    """
    gt_boxes = []
    if not os.path.exists(label_file_path):
        return gt_boxes

    with open(label_file_path, 'r') as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            class_id = int(parts[0])
            center_x, center_y, bbox_width, bbox_height = parts[1:]

            # Convert normalized YOLO format to absolute pixel coordinates (x1, y1, x2, y2)
            x1 = int((center_x - bbox_width / 2) * img_width)
            y1 = int((center_y - bbox_height / 2) * img_height)
            x2 = int((center_x + bbox_width / 2) * img_width)
            y2 = int((center_y + bbox_height / 2) * img_height)

            gt_boxes.append({'class_id': class_id, 'box': [x1, y1, x2, y2]})
    return gt_boxes

def visualize_detections(image_path, model, class_names_list, labels_dir, iou_threshold=0.5, conf_threshold=0.25):
    """
    Performs inference, classifies predictions, identifies false negatives, and draws
    bounding boxes with distinct colors and labels on the image.
    Returns the annotated image.
    """
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Could not load image {image_path}")
        return None

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_height, img_width, _ = image_rgb.shape

    # Construct path to ground truth label file
    base_name = os.path.basename(image_path).rsplit('.', 1)[0]
    label_file_path = os.path.join(labels_dir, f'{base_name}.txt')

    ground_truths = read_yolo_labels(label_file_path, img_width, img_height)

    # Perform inference
    results = model.predict(source=image_path, verbose=False, conf=conf_threshold, imgsz=640)

    predictions = []
    if results and len(results) > 0:
        for *xyxy, conf, cls in results[0].boxes.data.tolist():
            predictions.append({'class_id': int(cls), 'box': [int(x) for x in xyxy], 'confidence': conf})

    # --- Matching Logic (TP, FP, FN) ---
    matched_gt_indices = set()
    matched_pred_indices = set()
    true_positives = [] # {'gt_box': gt_box, 'pred_box': pred_box, 'confidence': conf}
    false_positives = [] # {'pred_box': pred_box, 'confidence': conf, 'class_id': class_id}
    false_negatives = [] # {'gt_box': gt_box, 'class_id': class_id}

    # Iterate through each ground truth box
    for i_gt, gt in enumerate(ground_truths):
        best_iou = 0.0
        best_pred_idx = -1

        # Find the best matching prediction for the current ground truth
        for i_pred, pred in enumerate(predictions):
            if i_pred in matched_pred_indices: # Skip already matched predictions
                continue

            if gt['class_id'] == pred['class_id']:
                iou = calculate_iou(gt['box'], pred['box'])
                if iou >= iou_threshold and iou > best_iou:
                    best_iou = iou
                    best_pred_idx = i_pred

        # If a match is found for the current ground truth
        if best_pred_idx != -1:
            matched_gt_indices.add(i_gt)
            matched_pred_indices.add(best_pred_idx)
            true_positives.append({
                'gt_box': ground_truths[i_gt]['box'],
                'pred_box': predictions[best_pred_idx]['box'],
                'confidence': predictions[best_pred_idx]['confidence'],
                'class_id': ground_truths[i_gt]['class_id']
            })

    # Identify False Positives (unmatched predictions)
    for i_pred, pred in enumerate(predictions):
        if i_pred not in matched_pred_indices:
            false_positives.append({
                'pred_box': pred['box'],
                'confidence': pred['confidence'],
                'class_id': pred['class_id']
            })

    # Identify False Negatives (unmatched ground truths)
    for i_gt, gt in enumerate(ground_truths):
        if i_gt not in matched_gt_indices:
            false_negatives.append({
                'gt_box': gt['box'],
                'class_id': gt['class_id']
            })

    # --- Draw Bounding Boxes ---
    annotated_image = image_rgb.copy()

    # Draw True Positives (Green)
    for tp in true_positives:
        label = f"{class_names_list[tp['class_id']]}: {tp['confidence']:.2f}"
        annotated_image = draw_bbox(annotated_image, tp['pred_box'], label, color=(0, 255, 0)) # Green

    # Draw False Positives (Red)
    for fp in false_positives:
        label = f"FP: {class_names_list[fp['class_id']]}: {fp['confidence']:.2f}"
        annotated_image = draw_bbox(annotated_image, fp['pred_box'], label, color=(255, 0, 0)) # Red

    # Draw False Negatives (Blue or Yellow for 'Missing')
    for fn in false_negatives:
        class_name = class_names_list[fn['class_id']]
        if 'missing' in class_name.lower():
            label = f"FN_Missing: {class_name}"
            annotated_image = draw_bbox(annotated_image, fn['gt_box'], label, color=(255, 255, 0)) # Yellow
        else:
            label = f"FN: {class_name}"
            annotated_image = draw_bbox(annotated_image, fn['gt_box'], label, color=(0, 0, 255)) # Blue

    return annotated_image

print("Visualization functions 'read_yolo_labels' and 'visualize_detections' defined.")

Visualization functions 'read_yolo_labels' and 'visualize_detections' defined.


In [ ]:
!pip install ultralytics --upgrade -q
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import yaml # Import the yaml library

# -----------------------------------------------------------------------------
# Visualization tuned for green PCBs (YOLO outputs in normalized xywh)
# Goals:
# - High-contrast colors against green solder mask
# - Less clutter: show text only for Missing/*, FNs, and low-confidence preds
# - Readable text: black outline + white text
# - Optional alpha-blended boxes for dense regions
# -----------------------------------------------------------------------------

# Text / drawing controls
FONT_SCALE_LABELS = 0.45
TEXT_THICKNESS = 1
BOX_THICKNESS = 3
BOX_THICKNESS_CRITICAL = 3

# Only show text when confidence is below this threshold (unless Missing/* or FN/FP label)
SHOW_TEXT_BELOW_CONF = 0.80

# Alpha blend factor for filled rectangles (0 = no fill, 0.25-0.45 recommended)
FILL_ALPHA = 0.30

# Abbreviations to reduce label length
CLASS_ABBREV = {
    "Button": "BTN",
    "Capacitor": "C",
    "Electrolytic Capacitor": "EC",
    "Resistor": "R",
    "Inductor": "L",
    "Ferrite Bead": "FB",
    "Diode": "D",
    "Zener Diode": "ZD",
    "Led": "LED",
    "IC": "IC",
    "Connector": "CONN",
    "Switch": "SW",
    "Test Point": "TP",
    "Pins": "PIN",
    "Pads": "PAD",
    "Clock": "CLK",
    "Display": "DISP",
    "Fuse": "FUSE",
    "Heatsink": "HS",
    "Potentiometer": "POT",
    "Jumper": "JMP",
    "EM": "EM",
}

def _is_missing(class_name: str) -> bool:
    return class_name.lower().startswith("missing")

def _abbr(class_name: str) -> str:
    return CLASS_ABBREV.get(class_name, class_name)

def _format_label(class_name: str, conf, prefix: str = "") -> str:
    base = _abbr(class_name)
    if prefix:
        base = f"{prefix}{base}"
    if conf is None:
        return base
    return f"{base} {conf:.2f}"

# Semantic, high-contrast colors (BGR or RGB? We're drawing on RGB arrays, but OpenCV expects BGR.
# NOTE: We draw using cv2 on RGB images here; color tuples will still work but channels are swapped.
# To keep things consistent with your current notebook (cv2 drawing on RGB arrays), we define colors as RGB.
# If you change to drawing on BGR, swap these.
COLOR_IDENTIFIED = (255, 0, 255)  # Magenta (complementary to green)
COLOR_CONNECTOR  = (255, 255, 0)  # Yellow
COLOR_MISSING    = (255, 0, 0)    # Red
COLOR_FP         = (255, 165, 0)  # Orange (false positive)
COLOR_FN         = (255, 0, 0)    # Red (false negative)
COLOR_NEUTRAL    = (255, 255, 255)

# Define the class color map (semantic, not per-class rainbow)
class_color_map = {
    # Identified components (magenta)
    'Button': COLOR_IDENTIFIED,
    'Capacitor': COLOR_IDENTIFIED,
    'Clock': COLOR_IDENTIFIED,
    'Diode': COLOR_IDENTIFIED,
    'Display': COLOR_IDENTIFIED,
    'EM': COLOR_IDENTIFIED,
    'Electrolytic Capacitor': COLOR_IDENTIFIED,
    'Ferrite Bead': COLOR_IDENTIFIED,
    'Fuse': COLOR_IDENTIFIED,
    'Heatsink': COLOR_IDENTIFIED,
    'IC': COLOR_IDENTIFIED,
    'Inductor': COLOR_IDENTIFIED,
    'Jumper': COLOR_IDENTIFIED,
    'Led': COLOR_IDENTIFIED,
    'Potentiometer': COLOR_IDENTIFIED,
    'Resistor': COLOR_IDENTIFIED,
    'Switch': COLOR_IDENTIFIED,
    'Test Point': COLOR_IDENTIFIED,
    'Transistor': COLOR_IDENTIFIED,
    'Zener Diode': COLOR_IDENTIFIED,

    # Connectors (yellow)
    'Connector': COLOR_CONNECTOR,

    # Neutral
    'Pads': COLOR_NEUTRAL,
    'Pins': COLOR_NEUTRAL,

    # Missing (red)
    'Missing Component': COLOR_MISSING,
    'Missing IC': COLOR_MISSING,
    'Missing Resistor': COLOR_MISSING,
    'Missing Capacitor': COLOR_MISSING,
    'Missing Diode': COLOR_MISSING,
    'Missing Ferrite Bead': COLOR_MISSING,
    'Missing Inductor': COLOR_MISSING,
    'Missing Led': COLOR_MISSING,
}

def _draw_text_with_outline(img, text, org, font_scale):
    # Black outline
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, font_scale,
                (0, 0, 0), thickness=TEXT_THICKNESS + 2, lineType=cv2.LINE_AA)
    # White fill
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, font_scale,
                (255, 255, 255), thickness=TEXT_THICKNESS, lineType=cv2.LINE_AA)

def draw_bbox(image, bbox, label="", color=(255, 0, 255),
              line_thickness=3, font_scale=0.5, fill_alpha=0.0):
    """Draw a bbox (normalized xywh) with optional alpha-filled background and readable text."""
    h, w, _ = image.shape
    x_center, y_center, bbox_w, bbox_h = bbox

    x_min = int((x_center - bbox_w / 2) * w)
    y_min = int((y_center - bbox_h / 2) * h)
    x_max = int((x_center + bbox_w / 2) * w)
    y_max = int((y_center + bbox_h / 2) * h)

    x_min = max(0, x_min); y_min = max(0, y_min)
    x_max = min(w - 1, x_max); y_max = min(h - 1, y_max)

    # Optional filled rectangle (alpha blend) for dense scenes
    if fill_alpha and fill_alpha > 0:
        overlay = image.copy()
        cv2.rectangle(overlay, (x_min, y_min), (x_max, y_max), color, thickness=-1)
        cv2.addWeighted(overlay, fill_alpha, image, 1 - fill_alpha, 0, dst=image)

    # Bounding box outline
    cv2.rectangle(image, (x_min, y_min), (x_max, y_max), color, line_thickness)

    # Label
    if label:
        (tw, th), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, TEXT_THICKNESS)
        # try above box; if not enough space, put below
        tx = x_min
        ty = y_min - 6
        if ty - th < 0:
            ty = y_min + th + 6
        _draw_text_with_outline(image, label, (tx, ty), font_scale)

    return image

def draw_true_positives(image, true_positives, class_names_list, class_color_map):
    annotated_image = image.copy()
    for tp in true_positives:
        class_name = class_names_list[tp['class_id']]
        color = class_color_map.get(class_name, COLOR_IDENTIFIED)

        # Reduce clutter: show label only if Missing/* or low confidence
        conf = float(tp['confidence'])
        show_text = _is_missing(class_name) or conf < SHOW_TEXT_BELOW_CONF
        label = _format_label(class_name, conf) if show_text else ""

        thickness = BOX_THICKNESS_CRITICAL if _is_missing(class_name) else BOX_THICKNESS
        fill = FILL_ALPHA if _is_missing(class_name) else 0.0

        annotated_image = draw_bbox(
            annotated_image, tp['pred_box'], label=label, color=color,
            line_thickness=thickness, font_scale=FONT_SCALE_LABELS, fill_alpha=fill
        )
    return annotated_image

def draw_false_positives(image, false_positives, class_names_list, class_color_map):
    annotated_image = image.copy()
    for fp in false_positives:
        class_name = class_names_list[fp['class_id']]
        conf = float(fp['confidence'])

        # Always label FP (so you can debug), but keep short
        label = _format_label(class_name, conf, prefix="FP ")
        annotated_image = draw_bbox(
            annotated_image, fp['pred_box'], label=label, color=COLOR_FP,
            line_thickness=BOX_THICKNESS, font_scale=FONT_SCALE_LABELS, fill_alpha=0.0
        )
    return annotated_image

def draw_false_negatives(image, false_negatives, class_names_list, class_color_map):
    annotated_image = image.copy()
    for fn in false_negatives:
        class_name = class_names_list[fn['class_id']]
        # Always label FN (critical)
        label = _format_label(class_name, None, prefix="FN ")
        thickness = BOX_THICKNESS_CRITICAL
        annotated_image = draw_bbox(
            annotated_image, fn['gt_box'], label=label, color=COLOR_FN,
            line_thickness=thickness, font_scale=FONT_SCALE_LABELS, fill_alpha=FILL_ALPHA
        )
    return annotated_image

def get_gt_annotations(label_file_path, img_width, img_height):
    """
    Reads ground truth annotations from a YOLO format .txt file.
    Returns a list of dictionaries, each with 'class_id', 'gt_box'.
    gt_box is in normalized [x_center, y_center, width, height] format.
    """
    gt_annotations = []
    if not os.path.exists(label_file_path):
        return gt_annotations

    with open(label_file_path, 'r') as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            class_id = int(parts[0])
            # YOLO format: class_id x_center y_center width height (normalized)
            gt_annotations.append({
                'class_id': class_id,
                'gt_box': parts[1:] # normalized x_c, y_c, w, h
            })
    return gt_annotations

def calculate_iou(boxA, boxB):
    """
    Calculates Intersection over Union (IoU) of two bounding boxes.
    Boxes are in normalized [x_center, y_center, width, height] format.
    """
    # Convert normalized xywh to xyxy
    xA1, yA1 = boxA[0] - boxA[2] / 2, boxA[1] - boxA[3] / 2
    xA2, yA2 = boxA[0] + boxA[2] / 2, boxA[1] + boxA[3] / 2
    xB1, yB1 = boxB[0] - boxB[2] / 2, boxB[1] - boxB[3] / 2
    xB2, yB2 = boxB[0] + boxB[2] / 2, boxB[1] + boxB[3] / 2

    # determine the coordinates of the intersection rectangle
    x_intersect_min = max(xA1, xB1)
    y_intersect_min = max(yA1, yB1)
    x_intersect_max = min(xA2, xB2)
    y_intersect_max = min(yA2, yB2)

    # compute the area of intersection rectangle
    inter_area = max(0, x_intersect_max - x_intersect_min) * max(0, y_intersect_max - y_intersect_min)

    # compute the area of both the prediction and ground-truth rectangles
    boxA_area = boxA[2] * boxA[3]
    boxB_area = boxB[2] * boxB[3]

    # compute the intersection over union by taking the intersection
    # area and dividing it by the sum of prediction + ground-truth
    # areas - the interesection area
    iou = inter_area / float(boxA_area + boxB_area - inter_area) if (boxA_area + boxB_area - inter_area) > 0 else 0
    return iou


def visualize_detections(img_path, model, class_names_list, labels_dir, iou_threshold=0.5):
    """
    Performs inference, compares with ground truth, and returns lists of
    True Positives, False Positives, and False Negatives.
    """
    true_positives = []
    false_positives = []
    false_negatives = []

    # Run inference
    results = model(img_path, verbose=False) # verbose=False to suppress extensive output
    predictions = results[0] # Get the first (and usually only) result object

    # Get image dimensions from the image itself
    img_temp = cv2.imread(img_path)
    if img_temp is None:
        print(f"Error: Could not load image {img_path} for GT processing.")
        return [], [], []
    img_height, img_width, _ = img_temp.shape

    # Get ground truth annotations
    base_img_name = os.path.basename(img_path).rsplit('.', 1)[0]
    label_file_path = os.path.join(labels_dir, f"{base_img_name}.txt")
    gt_annotations = get_gt_annotations(label_file_path, img_width, img_height)

    # Extract predictions from the model result
    pred_boxes_xywhn = predictions.boxes.xywhn.cpu().numpy() # normalized [x_center, y_center, width, height]
    pred_confs = predictions.boxes.conf.cpu().numpy()
    pred_class_ids = predictions.boxes.cls.cpu().numpy().astype(int)

    # Keep track of matched ground truth indices
    matched_gt_indices = [False] * len(gt_annotations)

    # Process predictions
    for i in range(len(pred_class_ids)):
        pred_box = pred_boxes_xywhn[i]
        pred_class_id = pred_class_ids[i]
        pred_conf = pred_confs[i]

        best_iou = 0
        best_gt_idx = -1

        # Find the best matching ground truth box for the current prediction
        for j, gt_ann in enumerate(gt_annotations):
            if matched_gt_indices[j]: # Skip already matched GTs
                continue

            gt_box = gt_ann['gt_box']
            iou = calculate_iou(pred_box, gt_box)

            if iou > best_iou:
                best_iou = iou
                best_gt_idx = j

        if best_iou >= iou_threshold and best_gt_idx != -1 and gt_annotations[best_gt_idx]['class_id'] == pred_class_id:
            # True Positive: Prediction matches a GT with sufficient IoU and correct class
            true_positives.append({
                'class_id': pred_class_id,
                'confidence': pred_conf,
                'pred_box': pred_box,
                'gt_box': gt_annotations[best_gt_idx]['gt_box'] # Include GT box for reference
            })
            matched_gt_indices[best_gt_idx] = True # Mark this GT as matched
        else:
            # False Positive: Prediction does not match any GT with sufficient IoU and correct class
            false_positives.append({
                'class_id': pred_class_id,
                'confidence': pred_conf,
                'pred_box': pred_box
            })

    # Identify False Negatives (unmatched ground truth boxes)
    for j, gt_ann in enumerate(gt_annotations):
        if not matched_gt_indices[j]:
            false_negatives.append({
                'class_id': gt_ann['class_id'],
                'gt_box': gt_ann['gt_box']
            })

    return true_positives, false_positives, false_negatives

def load_class_names_from_yaml(yaml_path):
    """
    Loads class names from a data.yaml file.
    """
    if not os.path.exists(yaml_path):
        print(f"Warning: data.yaml not found at {yaml_path}")
        return []
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    return data.get('names', [])

# Re-define ROOT_DIR if not already in scope, for robustness
if 'ROOT_DIR' not in globals():
    ROOT_DIR = '/content/drive/MyDrive/PCB_MC/Data/'

# Ensure VISUALIZATION_RESULTS_DIR is defined
if 'VISUALIZATION_RESULTS_DIR' not in globals():
    VISUALIZATION_RESULTS_DIR = '/content/yolov11_visualizations'
    os.makedirs(VISUALIZATION_RESULTS_DIR, exist_ok=True)

# Subsets for which to generate visualizations
subsets = ['components_only', 'full_dataset', 'missing_only', 'non_missing']

# Ensure results_dirs is accessible from the global scope and contains the fully corrected paths.
# This check ensures that the results_dirs dictionary is consistent with the latest update.
if 'results_dirs' not in globals() or not isinstance(results_dirs, dict) or \
   any(not path.startswith('/content/drive/MyDrive/PCB_MC/Results/YOLOV11/') for path in results_dirs.values()):
    results_dirs = {
        'components_only': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/components_only',
        'full_dataset': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/full_dataset',
        'missing_only': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/missing_only',
        'non_missing': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/non_missing'
    }
    print("Re-initialized results_dirs with fully corrected paths, assuming all models are in Results/YOLOV11.")
else:
    print("Using existing results_dirs with fully corrected paths.")


# 2. Select representative image file paths (modified logic here)
num_images_to_select = 12
all_subset_image_names = {}
for subset in subsets:
    images_path = os.path.join(ROOT_DIR, subset, 'kfold_data', 'fold_1', 'valid', 'images')
    if os.path.exists(images_path):
        all_subset_image_names[subset] = set([f for f in os.listdir(images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    else:
        all_subset_image_names[subset] = set()

# Find common image names across all subsets
if all_subset_image_names:
    common_image_names = set.intersection(*all_subset_image_names.values())
else:
    common_image_names = set()

selected_common_image_names = list(common_image_names)[:num_images_to_select]

selected_images_per_subset = {}
for subset in subsets:
    images_path = os.path.join(ROOT_DIR, subset, 'kfold_data', 'fold_1', 'valid', 'images')
    subset_images_to_process = []

    # Prioritize common images
    for img_name in selected_common_image_names:
        if img_name in all_subset_image_names[subset]:
            subset_images_to_process.append(os.path.join(images_path, img_name))

    # If not enough common images, fill with arbitrary images from the subset
    if len(subset_images_to_process) < num_images_to_select:
        available_images = [f for f in os.listdir(images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png')) and f not in selected_common_image_names]
        subset_images_to_process.extend([os.path.join(images_path, img_name) for img_name in available_images[:(num_images_to_select - len(subset_images_to_process))]])

    selected_images_per_subset[subset] = subset_images_to_process[:num_images_to_select] # Ensure max num_images_to_select
    print(f"Selected {len(selected_images_per_subset[subset])} images for {subset}")


# 3. Iterate through each subset and its selected images
for subset, image_paths in selected_images_per_subset.items():
    if not image_paths:
        print(f"No images to process for subset: {subset}")
        continue

    print(f"\nProcessing subset: {subset}")

    # a. Construct the full path to the best.pt weights for 'fold_0'
    weights_path = os.path.join(results_dirs[subset], 'fold_1', 'weights', 'best.pt')

    if not os.path.exists(weights_path):
        print(f"Weights not found for {subset} fold_1 at {weights_path}. Skipping.")
        continue

    # b. Load the YOLOv11 model
    print(f"Loading model for {subset} from {weights_path}")
    model = YOLO(weights_path)

    # c. Construct the path to the ground truth labels directory for 'fold_0'
    labels_dir = os.path.join(ROOT_DIR, subset, 'kfold_data', 'fold_1', 'valid', 'labels')
    if not os.path.exists(labels_dir):
        print(f"Labels directory not found for {subset} fold_1 at {labels_dir}. Skipping.")
        continue

    # d. Construct a unique output directory path for the current subset
    subset_output_dir = os.path.join(VISUALIZATION_RESULTS_DIR, subset)
    os.makedirs(subset_output_dir, exist_ok=True)
    print(f"Saving visualizations for {subset} to {subset_output_dir}")

    # Dynamically load class names for the current subset from its data.yaml
    data_yaml_path = os.path.join(ROOT_DIR, subset, 'kfold_data', 'fold_0', 'data.yaml')
    current_class_names = load_class_names_from_yaml(data_yaml_path)

    if not current_class_names:
        print(f"Could not load class names from {data_yaml_path}. Skipping visualization for {subset}.")
        continue

    # e. For each selected image, call the visualize_detections function
    for img_path in image_paths:
        print(f"  Visualizing: {os.path.basename(img_path)}")
        original_image = cv2.imread(img_path)
        if original_image is None:
            print(f"Error: Could not load image {img_path}")
            continue
        original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)

        # Get TP, FP, FN lists
        true_positives, false_positives, false_negatives = visualize_detections(
            img_path, model, current_class_names, labels_dir
        )

        # Create copies of the original image for drawing
        img_tp = original_image_rgb.copy()
        img_fp = original_image_rgb.copy()
        img_fn = original_image_rgb.copy()

        # Draw bounding boxes for each detection type
        img_tp = draw_true_positives(img_tp, true_positives, current_class_names, class_color_map)
        img_fp = draw_false_positives(img_fp, false_positives, current_class_names, class_color_map)
        img_fn = draw_false_negatives(img_fn, false_negatives, current_class_names, class_color_map)

        # Save the annotated images
        base_img_name = os.path.basename(img_path).rsplit('.', 1)[0]

        output_filepath_tp = os.path.join(subset_output_dir, f"{base_img_name}_TP.jpg")
        cv2.imwrite(output_filepath_tp, cv2.cvtColor(img_tp, cv2.COLOR_RGB2BGR))
        print(f"    Saved True Positive visualization to {output_filepath_tp}")

        output_filepath_fp = os.path.join(subset_output_dir, f"{base_img_name}_FP.jpg")
        cv2.imwrite(output_filepath_fp, cv2.cvtColor(img_fp, cv2.COLOR_RGB2BGR))
        print(f"    Saved False Positive visualization to {output_filepath_fp}")

        output_filepath_fn = os.path.join(subset_output_dir, f"{base_img_name}_FN.jpg")
        cv2.imwrite(output_filepath_fn, cv2.cvtColor(img_fn, cv2.COLOR_RGB2BGR))
        print(f"    Saved False Negative visualization to {output_filepath_fn}")

print("\nImage visualization process complete!")

Using existing results_dirs with fully corrected paths.
Selected 12 images for components_only
Selected 12 images for full_dataset
Selected 12 images for missing_only
Selected 12 images for non_missing

Processing subset: components_only
Loading model for components_only from /content/drive/MyDrive/PCB_MC/Results/YOLOV11/components_only/fold_1/weights/best.pt
Saving visualizations for components_only to /content/yolov11_visualizations/components_only
  Visualizing: ACM-109_Bottom_jpg.rf.14d920b31fdb4219b606465e90e42396.jpg
    Saved True Positive visualization to /content/yolov11_visualizations/components_only/ACM-109_Bottom_jpg.rf.14d920b31fdb4219b606465e90e42396_TP.jpg
    Saved False Positive visualization to /content/yolov11_visualizations/components_only/ACM-109_Bottom_jpg.rf.14d920b31fdb4219b606465e90e42396_FP.jpg
    Saved False Negative visualization to /content/yolov11_visualizations/components_only/ACM-109_Bottom_jpg.rf.14d920b31fdb4219b606465e90e42396_FN.jpg
  Visualizing: A